# Lesson 3: Classes

Enough object-oriented programming to read and write a real HEP analysis.

<img src="../img/map-3.svg" width="85%" alt="">

## Why classes

You have been using classes all along: every array, histogram and figure was defined as a class by some library.

Project 2 kept the data (arrays) in one place and the behaviour (functions) in another, and you passed one to the other by hand. A class keeps them together and gives the combination a name.

It is also the shape of the analysis tools in this school: to a framework, *your analysis* is a class that it can configure, split and run for you.

## Objects

Everything in Python is an object: a value that carries data and methods.

A method is a function attached to a value. You call it with a dot.

In [ ]:
import numpy as np

pt = np.array([63.0, 38.1, 4.0])
pt.max(), pt.shape, "muon".upper()

## Classes

A class is a recipe. Objects made from it each have their own data.

<img src="../img/class-objects.svg" width="85%" alt="">

## Everything you used was a class

`type` from lesson 1 names the class that made each value. Arrays, histograms and even floats are objects of classes that someone defined.

In [ ]:
import hist

type(pt), type(63.0), type(hist.Hist.new.Reg(10, 0, 1).Double())

## `__init__` and `self`

`__init__` sets up a new object. `self` is the object being built.

In [ ]:
class Muon:
    def __init__(self, pt, charge):
        self.pt = pt
        self.charge = charge

mu = Muon(63.0, 1)
mu.pt

## Separate data

Each object keeps its own data.

In [ ]:
other = Muon(38.1, -1)
mu.pt, other.pt

## Methods

A method is a function defined inside the class. Its first argument is always `self`.

In [ ]:
class Muon:
    def __init__(self, pt, charge):
        self.pt = pt
        self.charge = charge

    def opposite(self, other):
        return self.charge != other.charge

Muon(63.0, 1).opposite(Muon(38.1, -1))

## Printing your objects

Without help, Python prints `<__main__.Muon object at 0x10...>`. `__repr__` returns the text to show instead. Write it once, read it everywhere.

In [ ]:
class Muon:
    def __init__(self, pt, charge):
        self.pt = pt
        self.charge = charge

    def __repr__(self):
        return f"Muon(pt={self.pt}, charge={self.charge})"


Muon(63.0, 1)

## Help your future self

A string right under `def` is the docstring. `help` and the question mark show it back.

In [ ]:
def add(p, q):
    """Component-wise sum of two four-vectors."""
    ...


help(add)

## Puzzle: who passed the argument?

```python
class Muon:
    def flipped():
        return ...

mu.flipped()
```

`TypeError: flipped() takes 0 positional arguments but 1 was given.`

You passed nothing. Who passed the argument?

<img src="../img/self-call.svg" width="75%" alt="">


Python did: `mu.flipped()` means `Muon.flipped(mu)`. The missing parameter is `self`. Every method takes it first.

## Properties

`@property` makes a method read like data: `z.mass`, with no brackets.

This is how `vector` gives you `.mass` in lesson 4.

In [ ]:
class FourVector:
    def __init__(self, E, px, py, pz):
        self.E = E
        self.px = px
        self.py = py
        self.pz = pz

    @property
    def mass(self):
        return np.sqrt(self.E**2 - self.px**2 - self.py**2 - self.pz**2)

FourVector(5.0, 1.0, 2.0, 2.0).mass

## Puzzle: assigning to a property

`z.mass` is computed from the four components. What happens on

```python
z.mass = 999.0
```

Guess before you run.

In [ ]:
z = FourVector(91.2, 0.0, 0.0, 0.0)
z.mass = 999.0

A property without a setter cannot be assigned. Your objects can refuse nonsense: nobody can hand this analysis an inconsistent four-vector.

## `__add__`

`__add__` tells Python what `+` means for your objects.

Names with two underscores on each side are special methods. Python calls them for you.

In [ ]:
class FourVector:
    def __init__(self, E, px, py, pz):
        self.E = E
        self.px = px
        self.py = py
        self.pz = pz

    def __add__(self, other):
        return FourVector(
            self.E + other.E,
            self.px + other.px,
            self.py + other.py,
            self.pz + other.pz,
        )

    @property
    def mass(self):
        return np.sqrt(self.E**2 - self.px**2 - self.py**2 - self.pz**2)

(FourVector(5.0, 1.0, 2.0, 2.0) + FourVector(5.0, -1.0, -2.0, -2.0)).mass

## Operators are method calls

`a + b` runs `a.__add__(b)`. `len(x)` runs `x.__len__()`. `a > b` runs `a.__gt__(b)`. The double underscores mark the methods that Python itself calls.

That is the whole trick behind lesson 2: NumPy wrote `__gt__` to return a mask.

In [ ]:
pt.__gt__(20.0)

`==` is `__eq__`. Without it, `==` means "the very same object":

In [ ]:
Muon(63.0, 1) == Muon(63.0, 1)

Two equal muons compare as `False` until you define `__eq__`. Every operator is yours to define, and only the ones you define exist.

## Why you can chain dots

Lesson 2 built histograms with `hist.Hist.new.Reg(50, 0, 100).Double()`. Each piece returns an object, and the next dot asks that object. A chain is nothing but "ask the result again".

In [ ]:
import hist

type(hist.Hist.new), type(hist.Hist.new.Reg(50, 0, 100))

## Columns of particles

The same class works for one particle or for a whole column of particles.

If the data are NumPy arrays, every method works element by element.

In [ ]:
E = np.array([5.0, 10.0])
px = np.array([1.0, 6.0])
zeros = np.zeros(2)
FourVector(E, px, zeros, zeros).mass

## Settings

A class can hold settings that its methods use.

In [ ]:
class Selection:
    def __init__(self, min_pt):
        self.min_pt = min_pt

    def apply(self, pt):
        return pt[pt > self.min_pt]

Selection(min_pt=20).apply(pt)

## Inheritance

A new class starts from an existing one and changes only what differs.

`super().__init__(...)` runs the parent's `__init__`.

In [ ]:
class TightSelection(Selection):
    def __init__(self):
        super().__init__(min_pt=50)

TightSelection().apply(pt)

## `isinstance`

An object of a child class still counts as the parent: code written for `Selection` accepts a `TightSelection` without knowing it exists. That is the point of inheritance.

In [ ]:
isinstance(TightSelection(), Selection)

## Processors

In HEP analysis frameworks such as coffea, an analysis is a class with a `process(self, events)` method.

You write the method. The framework splits the data into chunks, calls `process` on each chunk and adds up the outputs.

<img src="../img/process-chunks.svg" width="85%" alt="">

## A processor in plain Python

Settings go in `__init__`, the analysis in `process`.

In [ ]:
class CountProcessor:
    def __init__(self, min_pt):
        self.min_pt = min_pt

    def process(self, events):
        return {"events": len(events), "passed": int((events > self.min_pt).sum())}

processor = CountProcessor(min_pt=20)
processor.process(np.array([63.0, 38.1, 4.0]))

## Adding outputs

Outputs from separate chunks add up, key by key.

<img src="../img/two-ways.svg" width="80%" alt="">


In [ ]:
chunks = [np.array([63.0, 38.1, 4.0]), np.array([54.3, 23.5]), np.array([12.0])]
total = processor.process(chunks[0])
for chunk in chunks[1:]:
    output = processor.process(chunk)
    total = {key: total[key] + output[key] for key in total}
total

## In real code: `@dataclass`

When a class is mostly stored data, Python writes `__init__` and `__repr__` for you. You wrote them by hand today so that nothing in them is magic.

In [ ]:
from dataclasses import dataclass

@dataclass
class Muon:
    pt: float
    charge: int


Muon(63.0, 1)

## What you can now read

| You see | It means |
|---|---|
| `class Muon:` | define a recipe for objects |
| `def __init__(self, ...):` | set up a new object |
| `self.pt = pt` | store data on the object |
| `mu.pt`, `mu.opposite(other)` | read data, call a method |
| `@property` | a method you read without brackets |
| `__add__` | what `+` means for your objects |
| `class B(A):`, `super()` | start from an existing class |
| `isinstance(x, Selection)` | is `x` a `Selection`, or a child of it? |
| `def __repr__(self):` | how the object prints |
| `@dataclass` | `__init__` and `__repr__`, written for you |


## Now: project 3.

[project-higgs.ipynb](project-higgs.ipynb): a four-vector class and a Higgs processor.

<img src="../img/class-objects.svg" width="70%" alt="">